# IBD - Lab 07
## Market-Basket: Frequent Items and Association Rules


### Configuración

Configuremos Spark en su entorno de Colab. ¡Ejecuta la celda de abajo!

In [ ]:
# 1) Clean install
# !pip uninstall -y pyspark findspark -q
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install -q pyspark
!pip install -q gdown


In [ ]:
# 2) Set Java path
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Optional: check Java
!java -version
!echo $JAVA_HOME

Ahora autenticamos un cliente de Google Drive para descargar el archivo que procesaremos en nuestro trabajo de Spark.

**Asegúrese de seguir las instrucciones interactivas.**

In [ ]:
!pip install PyDrive
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:

!gdown --id 1dhi1F78ssqR8gE6U-AgB80ZW7V_9snX4 -O products.csv
!gdown --id 1KZBNEaIyMTcsRV817us6uLZgm-Mii8oU -O order_products__train.csv




Si ejecutó las celdas anteriores, debería poder ver el conjunto de datos que necesitaremos para este Colab en la pestaña "Archivos" en el panel izquierdo.

In [ ]:
# 3) Start Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("firstspark")
    .master("local[*]")
    .config("spark.ui.port", "4050")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark version:", spark.version)
print("SparkContext OK:", sc.appName)

Puede verificar fácilmente la versión actual y obtener el enlace a la interfaz web. En la interfaz de usuario de Spark, puede monitorear el progreso de su trabajo y depurar los cuellos de botella de rendimiento (si su Colab se ejecuta con un **tiempo de ejecución local**).

In [ ]:
spark

### Tu tarea

Si ejecuta correctamente la etapa de configuración, estará listo para trabajar con el conjunto de datos de **3 millones de pedidos de Instacart**. En caso de que desee leer más al respecto, consulte la [publicación del blog oficial de Instacart] (https://tech.instacart.com/3-million-instacart-orders-open-sourced-d40d29ead6f2) al respecto, un [esquema descripción](https://gist.github.com/jeremystan/c3b39d947d9b88b3ccff3147dbcf6c6b) del conjunto de datos y la [página de descarga](https://www.instacart.com/datasets/grocery-shopping-2017).

En este Colab, trabajaremos solo con un pequeño conjunto de datos de entrenamiento (~131 000 pedidos) para realizar una minería de patrones frecuentes rápida con el algoritmo FP-Growth.

Cargue los ficheros products.csv y order_products__train.csv con Spark DataFrames.


In [7]:
# YOUR
products = spark.read.csv('products.csv', header=True, inferSchema=True)
orders = spark.read.csv('order_products__train.csv', header=True, inferSchema=True)

In [ ]:
products.printSchema()

In [ ]:
orders.printSchema()

Utilice la API Spark Dataframe para unir 'productos' y 'pedidos', de modo que pueda ver los nombres de los productos en cada transacción (y no solo sus ID). Luego, agrupe los pedidos por 'order_id' para obtener una fila por canasta (es decir, un conjunto de productos comprados juntos por un cliente).



In [ ]:
from pyspark.sql.functions import collect_list

# Join the two databases Orders and Products
transaction = orders.join(products, on='product_id').groupby('order_id').agg(collect_list('product_name').alias('products'))
transaction.take(5)

In [ ]:
# Let see the Schema of Transaction Data Frame
transaction.printSchema()

En este Colab exploraremos [MLlib](https://spark.apache.org/mllib/), la biblioteca escalable de aprendizaje automático de Apache Spark. Específicamente, puede usar su implementación del algoritmo [FP-Growth](https://spark.apache.org/docs/latest/ml-frequent-pattern-mining.html#fp-growth) para realizar de manera eficiente la minería de patrones frecuentes. en Chispa.
Use el ejemplo de Python en la documentación y entrene un modelo con

```minSupport=0.01``` y ```minConfidence=0.5```

In [13]:
# Importing the Frequent pattern minning library
from pyspark.ml.fpm import FPGrowth

fpGrowth = FPGrowth(itemsCol="products", minSupport=0.01, minConfidence=0.5)
# Creating the FPM model
model = fpGrowth.fit(transaction)

Calcule cuántos conjuntos de elementos frecuentes y reglas de asociación se generaron ejecutando FP-growth.

In [ ]:
# show the reults of FPgrowth model
model.freqItemsets.show()


In [ ]:
model.associationRules.show()

## ¿Qué ves en los resultados?

En el ejemplo anterior, al ejecutar FP-Growth con *minSupport = 0.01* y *minConfidence = 0.5*, no se generan reglas de asociación.

A partir de esta observación, realiza las siguientes tareas:

1- Ejecuta varios experimentos modificando los valores de minSupport y minConfidence.

2- Encuentra una configuración que sí genere reglas de asociación.
3- Compara al menos 3 configuraciones distintas de parámetros.

4- Para cada configuración, indica:
    a. número de itemsets frecuentes,
    b. número de reglas de asociación.

5- Selecciona la configuración que consideres más adecuada y justifica tu elección.

6- Muestra algunas de las reglas obtenidas con esa configuración e interpreta su significado.

Por ejemplo, puedes probar combinaciones como:

*minSupport: 0.01, 0.005, 0.001*

*minConfidence: 0.5, 0.3, 0.1*


In [ ]:
# YOUR CODE HERE





## Vamos a trazar un gráfico de barras con el número de frequent items:

Una vez seleccionada la combinación de parámetros que consideres más adecuada, utiliza los resultados del modelo FP-Growth para identificar los ítems más frecuentes.

1- Ordena los resultados por soporte de mayor a menor.

2- Selecciona los 20 ítems más frecuentes.

3- Genera un gráfico de barras mostrando dichos ítems y su soporte.

4-Finalmente, comenta brevemente qué productos aparecen con mayor frecuencia y si el resultado tiene sentido en el contexto del dataset.

In [ ]:
# YOUR CODE HERE




## Detene la sesión de Spark

In [ ]:
# YOUR CODE HERE
sc.stop()

Una vez que haya obtenido los resultados deseados, **diríjase a moodle y envíe su solución para este laboratorio**.